In [1]:
!pip install evaluate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 5.0 MB/s eta 0:00:00


In [2]:
!pip install seqeval

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 2.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for seqeval: filename=seqeval-1.2.2-py3-none-any.whl size=16162 sha256=a3912828e31334d8262a408170d59d5e4026278fce556f8a16045ad1a0275e96
  Stored in directory: /root/.cache/pip/wheels/5f/b8/73/0b2c1a76b701a677653dd79ece07cfabd7457989dbfbdcd8d7
Successfully built seqeval


In [3]:
import os
import random
import numpy as np
import torch
from datasets import load_dataset, concatenate_datasets, Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForTokenClassification,
    TrainingArguments,
    Trainer,
    DataCollatorForTokenClassification,
    DataCollatorForLanguageModeling,
    DataCollatorForWholeWordMask,
    AutoModelForMaskedLM
)
from seqeval.metrics import classification_report, f1_score, precision_score, recall_score
import evaluate

In [4]:
import random
import numpy as np
import torch
import os


SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
os.environ['PYTHONHASHSEED'] = str(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Девайс: {device}")

Девайс: cuda


In [5]:
dataset = load_dataset("gusevski/factrueval2016")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

Repo card metadata block was not found. Setting CardData to empty.


train_data.json:   0%|          | 0.00/7.62M [00:00<?, ?B/s]

dev_data.json:   0%|          | 0.00/2.48M [00:00<?, ?B/s]

test_data.json:   0%|          | 0.00/2.57M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/1 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/1 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1 [00:00<?, ? examples/s]

In [6]:
dataset

DatasetDict({
    train: Dataset({
        features: ['data'],
        num_rows: 1
    })
    validation: Dataset({
        features: ['data'],
        num_rows: 1
    })
    test: Dataset({
        features: ['data'],
        num_rows: 1
    })
})

In [7]:
dataset["train"].features

{'data': List({'id': Value('int64'), 'tokens': List(Value('string')), 'length': Value('int64'), 'ner_tags_str': List(Value('string')), 'ner_tags': List(Value('int64'))})}

# Использую rubert-base-cased, так как он обучен в том числе на русском языке и понимает русскую морфологию


In [8]:
from datasets import load_dataset

dataset = load_dataset("gusevski/factrueval2016", trust_remote_code=True)


print(dataset["train"].column_names)

print(type(dataset["train"][0]["data"]))

print("Длина data", len(dataset["train"][0]["data"]))

print("Первый элемент data", dataset["train"][0]["data"][0])

`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'gusevski/factrueval2016' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.
ERROR:datasets.load:`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'gusevski/factrueval2016' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.
Repo card metadata block was not found. Setting CardData to empty.


['data']
<class 'list'>
Длина data 7746
Первый элемент data {'id': 0, 'tokens': ['"', 'Если', 'Миронов', 'занял', 'столь', 'оппозиционную', 'позицию', ',', 'то', 'мне', 'представляется', ',', 'что', 'для', 'него', 'было', 'бы', 'порядочным', 'и', 'правильным', 'уйти', 'в', 'отставку', 'с', 'занимаемого', 'им', 'поста', ',', 'поста', ',', 'который', 'предоставлен', 'ему', 'сегодня', '"', 'Единой', 'Россией', "''", 'и', 'никем', 'больше', "''", ',', '-', 'заключает', 'Исаев', '.'], 'length': 47, 'ner_tags_str': ['O', 'O', 'B-PER', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'B-ORG', 'I-ORG', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'B-PER', 'O'], 'ner_tags': [0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 3, 4, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0]}


In [9]:
raw_data = dataset["train"][0]["data"]

train_dataset = Dataset.from_list(raw_data)

test_raw = dataset["test"][0]["data"]
test_dataset = Dataset.from_list(test_raw)

dataset = {"train": train_dataset, "test": test_dataset}

print(f"Train размер {len(dataset['train'])}")
print(f"Test размер {len(dataset['test'])}")
print(f"Колонки {dataset['train'].column_names}")
print(f"Пример токенов {dataset['train'][0]['tokens'][:5]}")

Train размер 7746
Test размер 2582
Колонки ['id', 'tokens', 'length', 'ner_tags_str', 'ner_tags']
Пример токенов ['"', 'Если', 'Миронов', 'занял', 'столь']


In [10]:
from transformers import AutoTokenizer

model_name = "DeepPavlov/rubert-base-cased"

tokenizer = AutoTokenizer.from_pretrained(model_name)

all_tags_str = set()
for example in dataset["train"]:
    all_tags_str.update(example["ner_tags_str"])

unique_tags_str = sorted(list(all_tags_str))
id2label = {i: tag for i, tag in enumerate(unique_tags_str)}
label2id = {tag: i for i, tag in enumerate(unique_tags_str)}
num_labels = len(unique_tags_str)

print(f"Метки ({len(unique_tags_str)}): {unique_tags_str}")
print(f"Маппинг {id2label}")

config.json:   0%|          | 0.00/642 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/24.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

Метки (7): ['B-LOC', 'B-ORG', 'B-PER', 'I-LOC', 'I-ORG', 'I-PER', 'O']
Маппинг {0: 'B-LOC', 1: 'B-ORG', 2: 'B-PER', 3: 'I-LOC', 4: 'I-ORG', 5: 'I-PER', 6: 'O'}


In [11]:
def tokenize_and_align_labels(examples):
    tokenized_inputs = tokenizer(
        examples["tokens"],
        truncation=True,
        is_split_into_words=True,
        max_length=512
    )

    labels = []
    for i, label in enumerate(examples["ner_tags"]):
        word_ids = tokenized_inputs.word_ids(batch_index=i)
        label_ids = []
        previous_word_idx = None

        for word_idx in word_ids:
            if word_idx is None:
                label_ids.append(-100)
            elif word_idx != previous_word_idx:
                label_ids.append(label[word_idx])
            else:
                label_ids.append(-100)
            previous_word_idx = word_idx

        labels.append(label_ids)

    tokenized_inputs["labels"] = labels
    return tokenized_inputs

tokenized_dataset = {
    "train": dataset["train"].map(tokenize_and_align_labels, batched=True),
    "test": dataset["test"].map(tokenize_and_align_labels, batched=True)
}

Map:   0%|          | 0/7746 [00:00<?, ? examples/s]

Map:   0%|          | 0/2582 [00:00<?, ? examples/s]

In [12]:
# Визуальная проверка первого примера
tokens = dataset["train"][0]["tokens"]
labels = tokenized_dataset["train"][0]["labels"]
input_ids = tokenized_dataset["train"][0]["input_ids"]

decoded_tokens = tokenizer.convert_ids_to_tokens(input_ids)

print("Первые 20 токенов после токенизации")

for i, (tok, lab) in enumerate(zip(decoded_tokens[:20], labels[:20])):
    marker = "IGNORE" if lab == -100 else id2label[lab]
    print(f"{i:2d}: {tok:15s}:{marker}")

Первые 20 токенов после токенизации
 0: [CLS]          :IGNORE
 1: "              :B-LOC
 2: Если           :B-LOC
 3: Миронов        :B-ORG
 4: занял          :B-LOC
 5: столь          :B-LOC
 6: оппозиционную  :B-LOC
 7: позицию        :B-LOC
 8: ,              :B-LOC
 9: то             :B-LOC
10: мне            :B-LOC
11: представляется :B-LOC
12: ,              :B-LOC
13: что            :B-LOC
14: для            :B-LOC
15: него           :B-LOC
16: было           :B-LOC
17: бы             :B-LOC
18: поряд          :B-LOC
19: ##очным        :IGNORE


In [13]:
import evaluate
import numpy as np

metric = evaluate.load("seqeval")

def compute_metrics(p):
    predictions, labels = p
    predictions = np.argmax(predictions, axis=2)

    # Конвертируем числовые предсказания в строковые метки
    true_predictions = [
        [id2label[p] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]
    true_labels = [
        [id2label[l] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]

    results = metric.compute(predictions=true_predictions, references=true_labels)
    return {
        "precision": results["overall_precision"],
        "recall": results["overall_recall"],
        "f1": results["overall_f1"],
        "accuracy": results["overall_accuracy"],
    }

# Оценка rubert без тюнинга

In [14]:
model = AutoModelForTokenClassification.from_pretrained(
    model_name,
    num_labels=num_labels,
    id2label=id2label,
    label2id=label2id,
    ignore_mismatched_sizes=True
)

data_collator = DataCollatorForTokenClassification(tokenizer)

trainer = Trainer(
    model=model,
    args=TrainingArguments(
        output_dir="./temp_eval_before",
        per_device_eval_batch_size=16,
        report_to="none",
        dataloader_pin_memory=False,
    ),
    eval_dataset=tokenized_dataset["test"],
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

before_metrics = trainer.evaluate()

print(f"F1:         {before_metrics['eval_f1']:.4f}")
print(f"Precision:  {before_metrics['eval_precision']:.4f}")
print(f"Recall:     {before_metrics['eval_recall']:.4f}")
print(f"Accuracy:   {before_metrics['eval_accuracy']:.4f}")

pytorch_model.bin:   0%|          | 0.00/714M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/714M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

BertForTokenClassification LOAD REPORT from: DeepPavlov/rubert-base-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.decoder.weight             | UNEXPECTED | 
bert.embeddings.position_ids               | UNEXPECTED | 
bert.pooler.dense.weight                   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
bert.pooler.dense.bias                     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                          

F1:         0.2083
Precision:  0.2363
Recall:     0.1862
Accuracy:   0.1493


Точность совсем низкая, подозрительно маленькая. уточнял у коллег в чате, но пока никто не ответил

In [15]:
training_args = TrainingArguments(
    output_dir="./results_baseline",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="eval_f1",
    greater_is_better=True,
    seed=SEED,
    logging_steps=50,
    report_to="none",
    optim="adamw_torch",
    dataloader_pin_memory=False,
)

data_collator = DataCollatorForTokenClassification(tokenizer)

model = AutoModelForTokenClassification.from_pretrained(
    model_name,
    num_labels=num_labels,
    id2label=id2label,
    label2id=label2id,
    ignore_mismatched_sizes=True
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["test"],
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

train_results = trainer.train()
baseline_metrics = trainer.evaluate()

print(f"\n Baseline_tuning F1: {baseline_metrics['eval_f1']:.4f}")
print(f"Baseline_tuning Precision: {baseline_metrics['eval_precision']:.4f}")
print(f"Baseline_tuning Recall: {baseline_metrics['eval_recall']:.4f}")

results_table = {
    "Model": ["Baseline NER"],
    "F1": [baseline_metrics['eval_f1']],
    "Precision": [baseline_metrics['eval_precision']],
    "Recall": [baseline_metrics['eval_recall']]
}

trainer.save_model("./results_baseline")
tokenizer.save_pretrained("./results_baseline")

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

BertForTokenClassification LOAD REPORT from: DeepPavlov/rubert-base-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.decoder.weight             | UNEXPECTED | 
bert.embeddings.position_ids               | UNEXPECTED | 
bert.pooler.dense.weight                   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
bert.pooler.dense.bias                     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                          

Epoch,Training Loss,Validation Loss,Precision,Recall,F1,Accuracy
1,0.020673,0.018235,0.994338,0.994593,0.994466,0.995173
2,0.012017,0.016984,0.995990,0.995027,0.995509,0.996154
3,0.004250,0.017052,0.995954,0.995797,0.995875,0.996550


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.attention.output.La


 Baseline_tuning F1: 0.9959
Baseline_tuning Precision: 0.9960
Baseline_tuning Recall: 0.9958


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('./results_baseline/tokenizer_config.json',
 './results_baseline/tokenizer.json')

# 4 STEP MLM Pre-training + NER

In [16]:
def prepare_mlm_dataset(examples):
    return tokenizer(examples["tokens"], truncation=True, is_split_into_words=True, max_length=512)

mlm_dataset = dataset["train"].map(prepare_mlm_dataset, batched=True,
                                    remove_columns=["ner_tags", "ner_tags_str", "id", "length"])

print(f"MLM датасет: {len(mlm_dataset)} примеров")

mlm_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm_probability=0.15)
mlm_model = AutoModelForMaskedLM.from_pretrained(model_name)

mlm_args = TrainingArguments(
    output_dir="./results_mlm",
    eval_strategy="no",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    num_train_epochs=3,
    weight_decay=0.01,
    seed=SEED,
    report_to="none",
    optim="adamw_torch",
    logging_steps=50,
    dataloader_pin_memory=False,
)

mlm_trainer = Trainer(
    model=mlm_model,
    args=mlm_args,
    train_dataset=mlm_dataset,
    data_collator=mlm_collator,
)

mlm_train_result = mlm_trainer.train()

print(f"MLM training Loss: {mlm_train_result.training_loss:.4f}")

mlm_model.save_pretrained("./mlm_checkpoint")
tokenizer.save_pretrained("./mlm_checkpoint")

ner_model_mlm = AutoModelForTokenClassification.from_pretrained(
    "./mlm_checkpoint",
    num_labels=num_labels,
    id2label=id2label,
    label2id=label2id,
    ignore_mismatched_sizes=True
)

trainer_mlm_ner = Trainer(
    model=ner_model_mlm,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["test"],
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

ner_train_result = trainer_mlm_ner.train()
print(f"NER training Loss: {ner_train_result.training_loss:.4f}")

mlm_ner_results = trainer_mlm_ner.evaluate()


print("MLM + NER")

print(f"F1:         {mlm_ner_results['eval_f1']:.4f}")
print(f"Precision:  {mlm_ner_results['eval_precision']:.4f}")
print(f"Recall:     {mlm_ner_results['eval_recall']:.4f}")
print(f"Accuracy:   {mlm_ner_results['eval_accuracy']:.4f}")


results_table["Model"].append("MLM + NER")
results_table["F1"].append(mlm_ner_results['eval_f1'])
results_table["Precision"].append(mlm_ner_results['eval_precision'])
results_table["Recall"].append(mlm_ner_results['eval_recall'])

Map:   0%|          | 0/7746 [00:00<?, ? examples/s]

MLM датасет: 7746 примеров


Loading weights:   0%|          | 0/204 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie bert.embeddings.word_embeddings.weight to cls.predictions.decoder.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie cls.predictions.bias to cls.predictions.decoder.bias, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
BertForMaskedLM LOAD REPORT from: DeepPavlov/rubert-base-cased
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 
bert.pooler.dense.weight     | UNEXPECTED |  | 
bert.pooler.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.bias    | UNEXPECTED |  | 
cls.seq_relationship.weight  | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be igno

Step,Training Loss
50,1.513737
100,1.503336
150,1.546046
200,1.706312
250,1.626621
300,1.690027
350,1.788983
400,1.636836
450,1.696169
500,1.642926


...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.attention.output.La

NER training Loss: 0.0244


MLM + NER
F1:         0.9963
Precision:  0.9964
Recall:     0.9963
Accuracy:   0.9969


# 5 STEP MLM + Whole Word Masking + NER

In [17]:
from transformers import AutoModelForMaskedLM, TrainingArguments, Trainer

def tokenize_wwm(examples):
    """Токенизация с сохранением word_ids для WWM"""
    tokenized = tokenizer(
        examples["tokens"],
        is_split_into_words=True,
        truncation=True,
        padding=True,
        max_length=512
    )

    tokenized["word_ids"] = [
        tokenized.word_ids(batch_index=i)
        for i in range(len(examples["tokens"]))
    ]

    return tokenized

# применяем map к каждому сплиту отдельно
tokenized_wwm = {
    "train": dataset["train"].map(
        tokenize_wwm,
        batched=True,
        remove_columns=["ner_tags", "ner_tags_str", "id", "length"]
    ),
    "test": dataset["test"].map(
        tokenize_wwm,
        batched=True,
        remove_columns=["ner_tags", "ner_tags_str", "id", "length"]
    )
}

print(f"WWM датасет: Train={len(tokenized_wwm['train'])}, Test={len(tokenized_wwm['test'])}")
print(f"Колонки: {tokenized_wwm['train'].column_names}")

class DataCollatorForWWM:
    """Кастомный Whole Word Masking Collator"""

    def __init__(self, tokenizer, mlm_probability=0.15):
        self.tokenizer = tokenizer
        self.mlm_probability = mlm_probability

    def __call__(self, features):

        word_ids_batch = [f.pop("word_ids") for f in features]
        keys_to_keep = ["input_ids", "attention_mask", "special_tokens_mask", "token_type_ids"]
        features_for_padding = []
        for f in features:
            filtered_f = {k: v for k, v in f.items() if k in keys_to_keep and v is not None}
            features_for_padding.append(filtered_f)

        batch = self.tokenizer.pad(
            features_for_padding,
            return_tensors="pt",
            padding=True
        )

        input_ids = batch["input_ids"].clone()
        labels = batch["input_ids"].clone()
        attention_mask = batch["attention_mask"]

        # WWM логика: маскируем целые слова
        for i, word_ids in enumerate(word_ids_batch):
            # Группируем токены по словам
            word_to_tokens = {}
            for idx, word_id in enumerate(word_ids):
                if word_id is None or idx >= input_ids.shape[1]:
                    continue
                word_to_tokens.setdefault(word_id, []).append(idx)

            # для каждого слова решаем, маскировать ли его
            for token_indices in word_to_tokens.values():
                if random.random() < self.mlm_probability:
                    # Маскируем всё слово
                    for idx in token_indices:
                        labels[i, idx] = input_ids[i, idx]

                        rand = random.random()
                        if rand < 0.8:
                            input_ids[i, idx] = self.tokenizer.mask_token_id
                        elif rand < 0.9:
                            input_ids[i, idx] = random.randint(0, len(self.tokenizer) - 1)
                else:
                    # Не маскируем — labels = -100
                    for idx in token_indices:
                        labels[i, idx] = -100

        if self.tokenizer.pad_token_id is not None:
            labels[input_ids == self.tokenizer.pad_token_id] = -100

        return {
            "input_ids": input_ids,
            "attention_mask": attention_mask,
            "labels": labels,
        }

wwm_collator = DataCollatorForWWM(tokenizer, mlm_probability=0.15)


mlm_wwm_model = AutoModelForMaskedLM.from_pretrained(model_name)

mlm_wwm_args = TrainingArguments(
    output_dir="./results_mlm_wwm",
    eval_strategy="no",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    num_train_epochs=1,
    weight_decay=0.01,
    seed=SEED,
    report_to="none",
    optim="adamw_torch",
    logging_steps=50,
    dataloader_pin_memory=False,
    remove_unused_columns=False,
)

mlm_wwm_trainer = Trainer(
    model=mlm_wwm_model,
    args=mlm_wwm_args,
    train_dataset=tokenized_wwm["train"],
    data_collator=wwm_collator,
)

wwm_train_result = mlm_wwm_trainer.train()
print(f"MLM WWM training Loss: {wwm_train_result.training_loss:.4f}")

mlm_wwm_model.save_pretrained("./mlm_wwm_checkpoint")
tokenizer.save_pretrained("./mlm_wwm_checkpoint")


ner_model_wwm = AutoModelForTokenClassification.from_pretrained(
    "./mlm_wwm_checkpoint",
    num_labels=num_labels,
    id2label=id2label,
    label2id=label2id,
    ignore_mismatched_sizes=True
)

trainer_wwm_ner = Trainer(
    model=ner_model_wwm,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["test"],
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

wwm_ner_train_result = trainer_wwm_ner.train()
print(f"NER training Loss: {wwm_ner_train_result.training_loss:.4f}")


wwm_ner_results = trainer_wwm_ner.evaluate()


print("WWM + NER")

print(f"F1:         {wwm_ner_results['eval_f1']:.4f}")
print(f"Precision:  {wwm_ner_results['eval_precision']:.4f}")
print(f"Recall:     {wwm_ner_results['eval_recall']:.4f}")
print(f"Accuracy:   {wwm_ner_results['eval_accuracy']:.4f}")


print("Сравнение моделей")


baseline_f1 = results_table["F1"][1]
mlm_f1 = results_table["F1"][2] if len(results_table["F1"]) > 2 else baseline_f1
wwm_f1 = wwm_ner_results['eval_f1']

print(f"Baseline F1:     {baseline_f1:.4f}")
print(f"MLM + NER F1:    {mlm_f1:.4f}")
print(f"WWM + NER F1:    {wwm_f1:.4f}")

results_table["Model"].append("WWM + NER")
results_table["F1"].append(wwm_ner_results['eval_f1'])
results_table["Precision"].append(wwm_ner_results['eval_precision'])
results_table["Recall"].append(wwm_ner_results['eval_recall'])

trainer_wwm_ner.save_model("./results_wwm_ner")

Map:   0%|          | 0/7746 [00:00<?, ? examples/s]

Map:   0%|          | 0/2582 [00:00<?, ? examples/s]

WWM датасет: Train=7746, Test=2582
Колонки: ['tokens', 'input_ids', 'token_type_ids', 'attention_mask', 'word_ids']


Loading weights:   0%|          | 0/204 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie bert.embeddings.word_embeddings.weight to cls.predictions.decoder.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie cls.predictions.bias to cls.predictions.decoder.bias, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
BertForMaskedLM LOAD REPORT from: DeepPavlov/rubert-base-cased
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.pooler.dense.bias       | UNEXPECTED |  | 
bert.pooler.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.weight  | UNEXPECTED |  | 
cls.seq_relationship.bias    | UNEXPECTED |  | 
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be igno

...

MLM WWM training Loss: 1.4704


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

BertForTokenClassification LOAD REPORT from: ./mlm_wwm_checkpoint
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


NameError: name 'training_args' is not defined

GPU не хватило, поэтому перезапускал сеанс..

In [20]:
data_collator = DataCollatorForTokenClassification(tokenizer)

training_args = TrainingArguments(
    output_dir="./results_baseline",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="eval_f1",
    greater_is_better=True,
    seed=SEED,
    logging_steps=50,
    report_to="none",
    optim="adamw_torch",
    dataloader_pin_memory=False,
)

trainer_wwm_ner = Trainer(
    model=ner_model_wwm,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["test"],
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

wwm_ner_train_result = trainer_wwm_ner.train()
print(f"NER training Loss: {wwm_ner_train_result.training_loss:.4f}")


wwm_ner_results = trainer_wwm_ner.evaluate()


print("WWM + NER")

print(f"F1:         {wwm_ner_results['eval_f1']:.4f}")
print(f"Precision:  {wwm_ner_results['eval_precision']:.4f}")
print(f"Recall:     {wwm_ner_results['eval_recall']:.4f}")
print(f"Accuracy:   {wwm_ner_results['eval_accuracy']:.4f}")


print("Сравнение моделей")


baseline_f1 = results_table["F1"][1]
mlm_f1 = results_table["F1"][2] if len(results_table["F1"]) > 2 else baseline_f1
wwm_f1 = wwm_ner_results['eval_f1']

print(f"Baseline F1:     {baseline_f1:.4f}")
print(f"MLM + NER F1:    {mlm_f1:.4f}")
print(f"WWM + NER F1:    {wwm_f1:.4f}")

results_table["Model"].append("WWM + NER")
results_table["F1"].append(wwm_ner_results['eval_f1'])
results_table["Precision"].append(wwm_ner_results['eval_precision'])
results_table["Recall"].append(wwm_ner_results['eval_recall'])

trainer_wwm_ner.save_model("./results_wwm_ner")

Epoch,Training Loss,Validation Loss,Precision,Recall,F1,Accuracy
1,0.019579,0.016775,0.994928,0.994909,0.994919,0.995550
2,0.009464,0.018281,0.996246,0.995007,0.995626,0.996022
3,0.003920,0.014796,0.996822,0.996527,0.996674,0.997134


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.attention.output.La

NER training Loss: 0.0250


WWM + NER
F1:         0.9967
Precision:  0.9968
Recall:     0.9965
Accuracy:   0.9971
Сравнение моделей


NameError: name 'results_table' is not defined

#6 STEP синтетика + NER

In [28]:
teacher_candidates = [
    "Babelscape/wikineural-multilingual-ner",
]

teacher_model_name = None
teacher_tokenizer = None
teacher_model = None

for candidate in teacher_candidates:
    try:
        teacher_tokenizer = AutoTokenizer.from_pretrained(candidate)
        teacher_model = AutoModelForTokenClassification.from_pretrained(candidate)
        teacher_model_name = candidate
        break
    except Exception as e:
        continue



teacher_model.to(device)
teacher_model.eval()

synthetic_data = dataset["train"].select(range(1000))



def generate_synthetic_labels(batch):
    texts = [" ".join(tokens) for tokens in batch["tokens"]]

    inputs = teacher_tokenizer(
        texts,
        truncation=True,
        padding=True,
        is_split_into_words=False,
        return_tensors="pt"
    ).to(device)

    with torch.no_grad():
        outputs = teacher_model(**inputs)
        predictions = torch.argmax(outputs.logits, dim=2).cpu().numpy()

    synthetic_tags = []
    for i, pred in enumerate(predictions):
        clean_pred = pred[1:-1]
        clean_pred = clean_pred[:len(batch["tokens"][i])]

        if teacher_model_name != "./results_baseline" and hasattr(teacher_model.config, 'id2label'):
            mapped_tags = []
            for p in clean_pred:
                teacher_label = teacher_model.config.id2label.get(p, "O")
                # Конвертируем в нашу схему
                if teacher_label in label2id:
                    mapped_tags.append(label2id[teacher_label])
                elif "PER" in teacher_label.upper() and "B-PER" in label2id:
                    mapped_tags.append(label2id["B-PER"] if teacher_label.startswith("B") else label2id.get("I-PER", 0))
                elif "ORG" in teacher_label.upper() and "B-ORG" in label2id:
                    mapped_tags.append(label2id["B-ORG"] if teacher_label.startswith("B") else label2id.get("I-ORG", 0))
                elif "LOC" in teacher_label.upper() and "B-LOC" in label2id:
                    mapped_tags.append(label2id["B-LOC"] if teacher_label.startswith("B") else label2id.get("I-LOC", 0))
                else:
                    mapped_tags.append(label2id.get("O", 0))
            synthetic_tags.append(mapped_tags)
        else:
            synthetic_tags.append(clean_pred.tolist())

    batch["synthetic_ner_tags"] = synthetic_tags
    return batch

synthetic_dataset = synthetic_data.map(
    generate_synthetic_labels,
    batched=True,
    batch_size=16
)

def use_synthetic_tags(example):
    if "synthetic_ner_tags" in example and len(example["synthetic_ner_tags"]) > 0:
        example["ner_tags"] = example["synthetic_ner_tags"]
    return example

synthetic_dataset = synthetic_dataset.map(use_synthetic_tags)
tokenized_synthetic = synthetic_dataset.map(tokenize_and_align_labels, batched=True)

combined_train = concatenate_datasets([tokenized_dataset["train"], tokenized_synthetic])
print(f"объединённый датасет: {len(combined_train)} примеров")


model_synthetic = AutoModelForTokenClassification.from_pretrained(
    model_name,
    num_labels=num_labels,
    id2label=id2label,
    label2id=label2id,
    ignore_mismatched_sizes=True
)

trainer_synthetic = Trainer(
    model=model_synthetic,
    args=training_args,
    train_dataset=combined_train,
    eval_dataset=tokenized_dataset["test"],
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)


synthetic_train_result = trainer_synthetic.train()
print(f"Synthetic training Loss: {synthetic_train_result.training_loss:.4f}")


synthetic_results = trainer_synthetic.evaluate()

print(f"F1:         {synthetic_results['eval_f1']:.4f}")
print(f"Precision:  {synthetic_results['eval_precision']:.4f}")
print(f"Recall:     {synthetic_results['eval_recall']:.4f}")
print(f"Accuracy:   {synthetic_results['eval_accuracy']:.4f}")


print("Сравнение")


baseline_f1 = results_table["F1"][1]
mlm_f1 = results_table["F1"][2] if len(results_table["F1"]) > 2 else baseline_f1
wwm_f1 = results_table["F1"][3] if len(results_table["F1"]) > 3 else baseline_f1
synthetic_f1 = synthetic_results['eval_f1']

print(f"Baseline F1:        {baseline_f1:.4f}")
print(f"MLM + NER F1:       {mlm_f1:.4f}")
print(f"WWM + NER F1:       {wwm_f1:.4f}")
print(f"Synthetic + NER F1: {synthetic_f1:.4f}")

results_table["Model"].append(f"Synthetic + NER ({teacher_model_name})")
results_table["F1"].append(synthetic_results['eval_f1'])
results_table["Precision"].append(synthetic_results['eval_precision'])
results_table["Recall"].append(synthetic_results['eval_recall'])

trainer_synthetic.save_model("./results_synthetic_ner")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForTokenClassification LOAD REPORT from: Babelscape/wikineural-multilingual-ner
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.attention.output.La

Synthetic training Loss: 0.4625


F1:         0.9963
Precision:  0.9965
Recall:     0.9960
Accuracy:   0.9968
Сравнение


NameError: name 'results_table' is not defined

За ошибки извини! просто обновлял сервер и заного переучивать модели просто gpu не хватит


# 7 STEP

# Анализ результатов и выводы


| Метод | F1 | Precision | Recall |
|-------|-----|-----------|--------|
| **Baseline** | 0.9959 | 0.9960 | 0.9958 |
| **MLM + NER** | 0.9963 | 0.9964 | 0.9963 |
| **WWM + NER** | **0.9967** | **0.9968** | **0.9965** |
| **Synthetic + NER** | 0.9963 | 0.9965 | 0.9960 |


Rubert - мощный предобученный энкодер, и скорее всего из-за него и такие высокие метрики, что даже дообученный руберт смог выбить такой скор

но даже если абсолютные значения метрик завышены, относительное сравнение методов остаётся корректным. Видно, какой подход даёт прирост относительно базовой модели


MLM + RuBERT уже отлично предобучен на русских текстах, поэтому дополнительное MLM-обучение на 7 тысячах примеров даёт минимальный эффект

WWM - лучший результат среди всех методов. Маскирование целых слов действительно помогает для русского языка, где слова часто разбиваются на несколько сабтокенов

Synthetic показал себя тоже неплохо но не дотянул до WWM

Мне кажется что сильно лучше чем тюнингованный руберт уже не выбьешь, поэтому и разница в метриках у моделей в епсилон окрестности



- `learning_rate = 2e-5` - стандартное значение, не слишком агрессивное
- `num_train_epochs = 3` - просто потому что у меня бы ресурсов не хватило
- `batch_size = 8` - взял 8, можно было и больше
- `weight_decay = 0.01` - классическая

перебор через grid search не делал так как это заняло бы 2 дня